# Data Mining Project: From SVD Regression to Principal Component Regression

**Name:** ____________________ &nbsp;&nbsp;&nbsp; **Date:** ____________

## Scenario
A grain cooperative wants to measure the **protein content of wheat** without sending every sample to a slow, expensive chemistry lab. Instead, each sample is scanned with a **near-infrared (NIR) spectrometer**, which records how much light the sample absorbs at **90 wavelengths** between 1100 nm and 2500 nm. Your job is to build a model that predicts protein percentage from the spectrum.

## Learning objectives
By the end of this project you will be able to:
1. Solve a multiple linear regression problem using the **singular value decomposition (SVD)**.
2. Evaluate a regression model with **RMSE** and **R²** on training and held-out test data, and diagnose **overfitting**.
3. Explain how **multicollinearity** shows up in the singular values and in the fitted coefficients.
4. Perform **principal component analysis (PCA)** using the SVD.
5. Build a **principal component regression (PCR)** model, choose the number of components with **cross-validation**, and show that it generalizes better than ordinary least squares.

## The data: `wheat_nir_protein.csv`
| Column | Description |
|---|---|
| `sample_id` | Sample identifier (W001–W140) |
| `nm_1100` … `nm_2500` | Absorbance at each of 90 wavelengths (unitless) |
| `protein_pct` | Lab-measured protein content (% of dry mass), the **target** |

> **Note:** This is a simulated dataset built to behave like real NIR spectra (overlapping absorption bands from protein, moisture, starch, oil and fiber, plus baseline shifts and instrument noise). Keep the CSV in the same folder as this notebook.

## Instructions
* Complete every cell marked **`# TODO`**. Replace each `...` with your code.
* Answer every **📝 Reflection** question in the markdown cell provided (double-click to edit). Use specific numbers from *your* output to support your answers.
* Do **not** use `sklearn.linear_model.LinearRegression` or `sklearn.decomposition.PCA` for your models. The point is to build them yourself with `np.linalg.svd`. You may use scikit-learn only for `train_test_split` and `KFold`.
* Before submitting, choose **Kernel → Restart & Run All** and make sure the notebook runs top to bottom without errors.

## Suggested grading (100 pts)
| Part | Points |
|---|---|
| 1. Explore the data | 10 |
| 2. Train/test split and standardization | 5 |
| 3. Regression with the SVD | 20 |
| 4. Evaluate the SVD regression model | 15 |
| 5. PCA with the SVD | 15 |
| 6. Principal component regression | 20 |
| 7. Compare the models and conclude | 15 |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold

np.set_printoptions(precision=4, suppress=True)
plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42

---
## Part 1: Explore the data (10 pts)

Load the data and get a feel for it before modeling.

In [ ]:
df = pd.read_csv("wheat_nir_protein.csv")
df.head()

In [ ]:
# TODO: Build the feature matrix X (all the nm_ columns) and the target vector y (protein_pct)
#       as NumPy arrays. Also save the wavelengths as integers (e.g., 1100, 1115, ...) for plotting.
#       Hint: df.filter(like="nm_") selects the spectral columns.
feature_cols = ...
X = ...
y = ...
wavelengths = ...   # hint: [int(c.split("_")[1]) for c in feature_cols]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(f"Protein: mean = {y.mean():.2f}%, std = {y.std():.2f}%, min = {y.min():.2f}%, max = {y.max():.2f}%")

Run the cell below to plot a handful of spectra (one line per wheat sample), colored by protein content.

In [ ]:
fig, ax = plt.subplots()
idx = np.argsort(y)[::14]                       # every 14th sample, sorted by protein
cmap = plt.cm.viridis((y[idx] - y.min()) / (y.max() - y.min()))
for i, c in zip(idx, cmap):
    ax.plot(wavelengths, X[i], color=c, lw=1.2)
sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(y.min(), y.max()))
fig.colorbar(sm, ax=ax, label="protein (%)")
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("absorbance")
ax.set_title("Sample NIR spectra")
plt.show()

Next, look at how strongly the features are correlated with each other.

In [ ]:
corr = np.corrcoef(X, rowvar=False)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1,
               extent=[wavelengths[0], wavelengths[-1], wavelengths[-1], wavelengths[0]])
fig.colorbar(im, ax=ax, label="correlation")
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("wavelength (nm)")
ax.set_title("Correlation between wavelengths"); ax.grid(False)
plt.show()

# Correlation of each pair of NEIGHBORING wavelengths
neighbor_corr = np.array([corr[i, i + 1] for i in range(len(wavelengths) - 1)])
print(f"Median correlation between neighboring wavelengths: {np.median(neighbor_corr):.3f}")

In [ ]:
# TODO: Count the number of distinct feature PAIRS whose absolute correlation is greater than 0.9.
#       Hint: use np.triu_indices(p, k=1) to look at each pair only once.
p = X.shape[1]
n_pairs = ...
n_high = ...
print(f"{n_high} of {n_pairs} feature pairs ({100 * n_high / n_pairs:.1f}%) have |r| > 0.9")

### 📝 Reflection

**1.1 How many samples (n) and how many features (p) are there? Why might a ratio of n to p this close to 1 be a problem for multiple linear regression?**

*Your answer:*



**1.2 Describe what you see in the spectra plot and the correlation heatmap. What does it tell you about multicollinearity in this dataset? Use the numbers you printed.**

*Your answer:*



**1.3 Based only on this exploration, make a prediction: do you expect ordinary least squares on all 90 features to generalize well to new samples? Why or why not? (You will revisit this prediction later.)**

*Your answer:*

---
## Part 2: Train/test split and standardization (5 pts)

We hold out 30% of the samples as a **test set** that we will not touch until we evaluate a finished model.

Because PCA is sensitive to the scale of each feature, we **standardize** each column to mean 0 and standard deviation 1. The mean and standard deviation must be computed from the **training set only**, then applied to both the training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

In [ ]:
# TODO: Write a function that standardizes a training and test matrix using
#       the TRAINING mean and standard deviation (column by column).
def standardize(X_tr, X_te):
    mu = ...
    sd = ...
    Z_tr = ...
    Z_te = ...
    return Z_tr, Z_te

Z_train, Z_test = standardize(X_train, X_test)
print("Train column means (first 5):", Z_train.mean(axis=0)[:5])
print("Train column stds  (first 5):", Z_train.std(axis=0)[:5])
print("Test column means  (first 5):", Z_test.mean(axis=0)[:5], "<- close to 0, but not exactly")

### 📝 Reflection

**2.1 Why do we compute the mean and standard deviation from the training set only, instead of from all 140 samples? What would go wrong (or look misleadingly good) if we used all the data?**

*Your answer:*

---
## Part 3: Multiple linear regression with the SVD (20 pts)

We want coefficients $\boldsymbol\beta$ that minimize $\|\mathbf y - A\boldsymbol\beta\|^2$, where $A = [\mathbf 1 \;\; Z]$ is the standardized feature matrix with a column of ones added for the intercept.

Write the thin SVD $A = U\Sigma V^T$. The least-squares solution is

$$\boldsymbol\beta = V\,\Sigma^{-1}\,U^T\mathbf y \;=\; \sum_{i=1}^{r} \frac{\mathbf u_i^T\mathbf y}{\sigma_i}\,\mathbf v_i .$$

Keep that $1/\sigma_i$ in mind: it will matter a lot in Part 4.
* Note in the code that OLS stands for Ordinary Least Squares.

In [ ]:
# TODO: Complete the three functions below.
def add_intercept(Z):
    # Return [1  Z]: a column of ones followed by the columns of Z.
    return ...

def svd_regression(Z, y):
    # Fit least squares with the SVD. Return (beta, singular_values).
    # beta[0] is the intercept, beta[1:] are the feature coefficients.
    A = add_intercept(Z)
    U, s, Vt = ...                       # use np.linalg.svd(..., full_matrices=False)
    beta = ...                           # beta = V diag(1/s) U^T y
    return beta, s

def predict(Z, beta):
    return ...

beta_ols, sing_vals = svd_regression(Z_train, y_train)
print("Number of coefficients:", beta_ols.shape[0])
print("Intercept:", round(beta_ols[0], 4))

Check your work: your SVD solution should match NumPy's built-in least-squares solver.

In [ ]:
beta_check, *_ = np.linalg.lstsq(add_intercept(Z_train), y_train, rcond=None)
print("Max difference from np.linalg.lstsq:", np.max(np.abs(beta_ols - beta_check)))
assert np.allclose(beta_ols, beta_check, atol=1e-6), "Your SVD solution does not match. Check svd_regression()."
print("✅ SVD regression matches.")

In [ ]:
# TODO: Print the largest singular value, the smallest singular value, and the
#       condition number (largest / smallest) of the design matrix A.
#       Then plot all the singular values on a log-scale y-axis.
sigma_max = ...
sigma_min = ...
cond_A = ...
print(f"largest sigma = {sigma_max:.3f}, smallest sigma = {sigma_min:.5f}, condition number = {cond_A:,.0f}")

fig, ax = plt.subplots()
...
ax.set_yscale("log")
ax.set_xlabel("index i"); ax.set_ylabel(r"$\sigma_i$ (log scale)")
ax.set_title("Singular values of the design matrix")
plt.show()

### 📝 Reflection

**3.1 In your own words, explain each step of how the SVD produces the regression coefficients. Why is solving with the SVD more numerically stable than computing $(A^TA)^{-1}A^Ty$ directly? (Hint: how does the condition number of $A^TA$ compare to that of $A$?)**

*Your answer:*



**3.2 Look at the singular value plot. How many singular values are “large,” and roughly where does the curve drop off? What does a very small singular value say about the columns of $A$?**

*Your answer:*



**3.3 Using the formula $\boldsymbol\beta = \sum_i \frac{\mathbf u_i^T\mathbf y}{\sigma_i}\mathbf v_i$, explain what happens to the coefficients when some $\sigma_i$ are tiny and $\mathbf u_i^T\mathbf y$ is mostly noise.**

*Your answer:*

---
## Part 4: Evaluate the SVD regression model (15 pts)

We will use two metrics:

$$\text{RMSE} = \sqrt{\frac1n\sum_i (y_i-\hat y_i)^2}, \qquad R^2 = 1-\frac{\sum_i (y_i-\hat y_i)^2}{\sum_i (y_i-\bar y)^2}.$$

In [ ]:
# TODO: Implement RMSE and R^2 (do not use sklearn.metrics).
def rmse(y_true, y_pred):
    return ...

def r2(y_true, y_pred):
    return ...

# TODO: Compute predictions on the training and test sets and evaluate them.
yhat_train_ols = ...
yhat_test_ols = ...

ols_results = {
    "train RMSE": rmse(y_train, yhat_train_ols),
    "test RMSE":  rmse(y_test, yhat_test_ols),
    "train R2":   r2(y_train, yhat_train_ols),
    "test R2":    r2(y_test, yhat_test_ols),
}
pd.Series(ols_results, name="OLS (SVD, all 90 features)").round(4)

For context, here is a *baseline* that ignores the spectrum entirely and always predicts the training mean.

In [ ]:
baseline_rmse = rmse(y_test, np.full_like(y_test, y_train.mean()))
print(f"Baseline test RMSE (predict the mean): {baseline_rmse:.4f}")

In [ ]:
# TODO: Make a 1x2 figure:
#   left : predicted vs. actual protein for BOTH train (one color) and test (another color),
#          with a dashed y = x reference line.
#   right: the 90 feature coefficients beta_ols[1:] plotted against wavelength.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
...
ax.set_xlabel("actual protein (%)"); ax.set_ylabel("predicted protein (%)")
ax.set_title("OLS: predicted vs. actual"); ax.legend()

ax = axes[1]
...
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("coefficient")
ax.set_title("OLS coefficients")
plt.tight_layout(); plt.show()

### 📝 Reflection

**4.1 Compare the training and test RMSE and R². What is the size of the gap, and what does it tell you about this model? Is this overfitting or underfitting? Use your numbers.**

*Your answer:*



**4.2 How does the OLS test RMSE compare to the baseline that just predicts the mean? Is the model useful, and would you trust it to replace the chemistry lab?**

*Your answer:*



**4.3 Describe the coefficient plot. Do the coefficients look like a physically sensible relationship between absorbance and protein? Connect what you see to the small singular values and your answer to 3.3.**

*Your answer:*



**4.4 Go back to your prediction in 1.3. Was it correct? What evidence from Parts 3 and 4 supports or contradicts it?**

*Your answer:*

---
## Part 5: Principal component analysis with the SVD (15 pts)

Because `Z_train` is already centered, its SVD $Z = U S V^T$ gives us PCA directly:

* The **columns of $V$** (rows of `Vt`) are the principal directions, also called **loadings**.
* The **scores** (the data expressed in the new coordinates) are $T = ZV = US$.
* The **variance explained** by component $i$ is proportional to $s_i^2$, so the explained-variance ratio is $s_i^2 / \sum_j s_j^2$.

**Important:** do the PCA on the **training set only**, then project the test set with the same $V$.

In [ ]:
# TODO: Compute the SVD of Z_train (no intercept column here!) and the explained-variance ratios.
U_pca, s_pca, Vt_pca = ...
explained_ratio = ...
cumulative = ...

for k in [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]:
    print(f"First {k:2d} components explain {100 * cumulative[k - 1]:6.2f}% of the variance in X")

In [ ]:
# TODO: Make a scree plot: bar chart of explained_ratio for the first 20 components,
#       with the cumulative explained variance as a line on the same axes.
#       Also find the smallest k that explains at least 95% of the variance.
k_95 = ...
print("Components needed for 95% of the variance:", k_95)

fig, ax = plt.subplots()
...
ax.set_xlabel("principal component"); ax.set_ylabel("fraction of variance")
ax.set_title("Scree plot"); ax.legend()
plt.show()

Run the cell below to see what the first few principal directions look like as “spectra.”

In [ ]:
fig, ax = plt.subplots()
for i in range(4):
    ax.plot(wavelengths, Vt_pca[i], label=f"PC{i + 1} ({100 * explained_ratio[i]:.1f}%)")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("loading")
ax.set_title("First four principal directions (loadings)"); ax.legend()
plt.show()

### 📝 Reflection

**5.1 How many components are needed to explain 95% of the variance in the 90 features? What does this say about the “true” dimensionality of the data compared to the 90 measured wavelengths?**

*Your answer:*



**5.2 Compare the scree plot to the singular-value plot in Part 3. How are they related? What kind of information do you think lives in the last 60+ components?**

*Your answer:*



**5.3 The loadings are smooth curves rather than random noise. Why does that make sense for spectra? (Think about what physically causes the absorbance at each wavelength.)**

*Your answer:*

---
## Part 6: Principal component regression (20 pts)

**Principal component regression (PCR)** replaces the 90 correlated features with the first $k$ principal-component scores, then runs ordinary least squares on those scores:

1. Find the loadings $V_k$ (first $k$ columns of $V$) from the training data.
2. Compute scores $T_{\text{train}} = Z_{\text{train}} V_k$ and $T_{\text{test}} = Z_{\text{test}} V_k$.
3. Fit `svd_regression(T_train, y_train)` and predict with `predict(T_test, beta)`.

The big question is **how to choose $k$**. We must not choose it by looking at the test set (that would be cheating). Instead we use **5-fold cross-validation on the training set**.

In [ ]:
# TODO: Write a PCR function. Fit on (Z_tr, y_tr) using k components, and return predictions for Z_te.
def pcr_fit_predict(Z_tr, y_tr, Z_te, k):
    _, _, Vt = ...             # SVD of Z_tr
    Vk = ...                   # first k loading vectors as columns, shape (p, k)
    T_tr = ...
    T_te = ...
    beta, _ = svd_regression(T_tr, y_tr)
    return predict(T_tr, beta), predict(T_te, beta)

# quick sanity check: with k = p (all components), PCR should give the same predictions as OLS
_, check = pcr_fit_predict(Z_train, y_train, Z_test, Z_train.shape[1])
print("Max diff from OLS with k = p:", np.max(np.abs(check - yhat_test_ols)))

Now use 5-fold cross-validation **on the training set** to estimate the prediction error for each $k$. Inside each fold, standardize and do PCA using only that fold's training portion, exactly as we did for the full split.

In [ ]:
# TODO: For each k in k_values, compute the average validation RMSE across the 5 folds.
k_values = np.arange(1, 41)
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rmse = np.zeros(len(k_values))

for j, k in enumerate(k_values):
    fold_scores = []
    for tr_idx, val_idx in kf.split(X_train):
        # 1) standardize X_train[tr_idx] and X_train[val_idx] using the fold-training statistics
        # 2) run pcr_fit_predict with k components
        # 3) append the validation RMSE to fold_scores
        ...
    cv_rmse[j] = np.mean(fold_scores)

best_k = ...
print(f"Best k by 5-fold CV: {best_k}  (CV RMSE = {cv_rmse.min():.4f})")

In [ ]:
# TODO: For comparison, also compute the 5-fold CV RMSE of OLS on all features
#       (hint: PCR with k = number of features is the same as OLS).
cv_rmse_ols = ...
print(f"OLS 5-fold CV RMSE: {cv_rmse_ols:.4f}")

# TODO: Plot cv_rmse against k_values. Add a horizontal dashed line for the OLS CV RMSE
#       and mark best_k (and k_95 from Part 5) with vertical lines.
fig, ax = plt.subplots()
...
ax.set_xlabel("number of principal components k"); ax.set_ylabel("5-fold CV RMSE")
ax.set_title("Choosing k by cross-validation"); ax.legend()
plt.show()

Now that $k$ is chosen, fit the final PCR model on the **full training set** and evaluate it on the **test set** (for the first time).

In [ ]:
# TODO: Fit PCR with best_k on the full training set, then evaluate on train and test.
yhat_train_pcr, yhat_test_pcr = ...

pcr_results = {
    "train RMSE": ...,
    "test RMSE":  ...,
    "train R2":   ...,
    "test R2":    ...,
}
pd.Series(pcr_results, name=f"PCR (k = {best_k})").round(4)

Finally, run the cell below to see how *train* and *test* RMSE change with $k$. (We look at the test curve **only for reflection**, after $k$ was already chosen.)

In [ ]:
train_curve, test_curve = [], []
for k in k_values:
    yh_tr, yh_te = pcr_fit_predict(Z_train, y_train, Z_test, k)
    train_curve.append(rmse(y_train, yh_tr)); test_curve.append(rmse(y_test, yh_te))

fig, ax = plt.subplots()
ax.plot(k_values, train_curve, "o-", ms=3, label="train RMSE")
ax.plot(k_values, test_curve, "o-", ms=3, label="test RMSE")
ax.axhline(ols_results["test RMSE"], color="C3", ls="--", label="OLS test RMSE")
ax.axvline(best_k, color="C2", ls=":", label=f"chosen k = {best_k}")
ax.set_xlabel("number of principal components k"); ax.set_ylabel("RMSE")
ax.set_title("Bias–variance trade-off in PCR"); ax.legend()
plt.show()

### 📝 Reflection

**6.1 Describe the shape of the CV RMSE curve. Why is the error high for very small $k$? Why does it start to rise again as $k$ grows? Use the words *bias* and *variance*.**

*Your answer:*



**6.2 Compare the $k$ chosen by cross-validation with the $k$ needed to explain 95% of the variance (Part 5). Are they the same? Why might the components that explain the most variance in **X** not be the ones that best predict **y**? (Hint: look at the CV curve for $k$ = 1, 2, 3 versus $k$ = 4.)**

*Your answer:*



**6.3 Why is it important that we chose $k$ with cross-validation on the training set rather than by picking the $k$ with the lowest test RMSE in the last plot?**

*Your answer:*



**6.4 In the train/test plot, what happens to the *training* RMSE as $k$ increases? Why is training error alone a poor guide for choosing a model?**

*Your answer:*



**6.5 The CV curve is fairly flat over a wide range of $k$. Find the smallest $k$ whose CV RMSE is within about 5% of the minimum. Would you prefer that simpler model over the one with the absolute lowest CV RMSE? Explain the trade-off.**

*Your answer:*

---
## Part 7: Compare the models and conclude (15 pts)

In [ ]:
# TODO: Build a comparison table with one row per model (OLS and PCR) and columns:
#       number of predictors, train RMSE, test RMSE, train R2, test R2, CV RMSE.
#       Then print the percent reduction in test RMSE from OLS to PCR.
comparison = pd.DataFrame({
    ...
})
display(comparison.round(4))

pct_improvement = ...
print(f"PCR reduces test RMSE by {pct_improvement:.1f}% compared to OLS.")

In [ ]:
# TODO: Make a 1x2 figure of predicted vs. actual protein on the TEST set:
#       left = OLS, right = PCR. Use the same axis limits on both and include a y = x line.
#       Put the test RMSE in each subplot title.
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
...
plt.tight_layout(); plt.show()

Run the cell below to compare the PCR coefficients (mapped back to the 90 wavelengths) with the OLS coefficients.

In [ ]:
# PCR coefficients in the original (standardized) feature space: beta_features = V_k @ gamma
_, _, Vt_full = np.linalg.svd(Z_train, full_matrices=False)
Vk = Vt_full[:best_k].T
gamma, _ = svd_regression(Z_train @ Vk, y_train)
beta_pcr_features = Vk @ gamma[1:]

fig, ax = plt.subplots()
ax.plot(wavelengths, beta_ols[1:], "o-", ms=3, alpha=0.6, label="OLS")
ax.plot(wavelengths, beta_pcr_features, "o-", ms=3, label=f"PCR (k={best_k})")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("wavelength (nm)"); ax.set_ylabel("coefficient")
ax.set_title("Coefficients by wavelength"); ax.legend()
plt.show()
print(f"Largest |coef|: OLS = {np.abs(beta_ols[1:]).max():.2f}, PCR = {np.abs(beta_pcr_features).max():.2f}")

### 📝 Reflection

**7.1 Summarize the comparison. By how much did PCR improve the test RMSE and test R² over OLS? Did the train/test gap shrink? Quote your numbers.**

*Your answer:*



**7.2 Notice that PCR has a **higher** training error than OLS but a **lower** test error. Explain why this is not a contradiction.**

*Your answer:*



**7.3 Compare the OLS and PCR coefficient curves. Which is easier to interpret, and why does PCR produce smaller, smoother coefficients? Connect your answer to the $1/\sigma_i$ terms from Part 3.**

*Your answer:*



**7.4 PCR throws away most of the principal components. What information is being discarded, and why does discarding it help here? Is there a risk that PCR could throw away something important for predicting y?**

*Your answer:*



**7.5 Describe another real-world dataset (not spectra) where you would expect PCR to beat ordinary least squares. What features of that dataset make you think so? Then describe a dataset where PCR would probably **not** help.**

*Your answer:*



**7.6 If you were advising the grain cooperative, which model would you recommend and why? What else would you want to test before using it in production?**

*Your answer:*

---
## Optional extensions (extra credit)
* **Different splits:** Rerun the notebook with several values of `RANDOM_STATE`. Is PCR better than OLS every time? Report the mean and spread of the improvement.
* **Ridge regression via the SVD:** Ridge replaces $1/\sigma_i$ with $\sigma_i/(\sigma_i^2+\lambda)$. Implement it, choose $\lambda$ by cross-validation, and compare it with PCR. How is shrinking each component (ridge) different from keeping or dropping it (PCR)?
* **Supervised component selection:** Instead of keeping the first $k$ components, rank components by their correlation with `y_train` and keep the top $k$. Does that help? What is the danger?